[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C09_Reasoning_TTC_Course/03_verifiers_prm/03_verifiers_prm.ipynb)

# 03 · Verifier 与 PRM：结果监督 vs 过程监督，以及 Goodhart 过优化

<span style="background:#1f6feb;color:#fff;padding:2px 8px;border-radius:4px;font-size:12px">CPU</span> 纯 numpy + matplotlib 模拟，自包含，全部 cell 秒级跑完。

**本 notebook 你将完成：**

1. 构造"真实质量 $z$ + 带噪感知 + 可钻营伪特征"的模拟解答数据，**手写梯度下降**训练 logistic verifier；
2. 实现 **verifier-BoN** 并复现 **Goodhart 倒 U**：verifier 噪声小时 accuracy 随 $N$ 单调升，伪特征可被钻营时先升后降；
3. 对比四种选择器：**random / majority voting / verifier-BoN / oracle (pass@N)**；
4. 实现玩具 **PRM**：把解答建模为步骤序列，用 **Math-Shepherd 式 MC rollout** 自动造过程标签，对比 ORM 与 PRM(min 聚合) 的 BoN 差距；
5. 画 verifier 的**校准曲线**（reliability diagram）并计算 ECE；
6. 完成 **4 道 ✏️ 练习**（数值梯度校验、BoN 倒 U、Math-Shepherd 标签、PRM vs ORM）。

参考文献：Cobbe et al. 2021 (arXiv:2110.14168) · Lightman et al. 2023 *Let's Verify Step by Step* (arXiv:2305.20050) · Wang et al. 2023 *Math-Shepherd* (arXiv:2312.08935) · Gao et al. 2022 *Scaling Laws for Reward Model Overoptimization* (arXiv:2210.10760)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
plt.rcParams["font.sans-serif"] = ["Arial Unicode MS", "PingFang SC", "Hiragino Sans GB",
                                   "Noto Sans CJK SC", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

def sigmoid(t):
    return 1.0 / (1.0 + np.exp(-np.clip(t, -30, 30)))

print("numpy", np.__version__)

## 1 · 模拟世界 + 手写 logistic verifier

每道题采样 $M=64$ 条候选解答。每条解答有一个**不可观测的真实质量** $z$（决定真实对错：$z>0$ 即正确），
verifier 只能看到三个**可观测特征**：

| 特征 | 含义 | 与正确性的关系 |
|---|---|---|
| $x_1 = z + \varepsilon$ | 带噪的质量感知 | 强相关（verifier 的"真本事"） |
| $x_2 = 0.5\,c + \varepsilon$ | "步骤工整度"等表面特征 | 弱相关，**且生成器可以免费伪造** |
| $x_3 = \varepsilon$ | 纯噪声 | 无关 |

verifier 是一个手写梯度下降训练的 logistic regression：$v_\psi(x)=\sigma(w^\top x + b)$，
目标是二分类 BCE——这正是 Cobbe 2021 GSM8K verifier 的最简骨架（按**题目**划分 train/test，避免同题泄漏）。

In [ ]:
# ---------- 模拟数据 ----------
Q, M = 300, 64                                    # 300 道题 x 64 条候选解答
diff = rng.normal(0.8, 0.6, (Q, 1))               # 题目难度（题间差异 -> 每题 pass@1 不同）
z = rng.normal(0, 1, (Q, M)) - diff               # 真实质量（不可观测）
correct = (z > 0).astype(float)                   # 真实对错
print(f"平均 pass@1 = {correct.mean():.3f}")

# verifier 可见特征
x1 = z + rng.normal(0, 0.6, (Q, M))               # 带噪质量感知
x2 = 0.5 * correct + rng.normal(0, 1, (Q, M))     # 表面"工整度"特征（弱相关、可伪造）
x3 = rng.normal(0, 1, (Q, M))                     # 纯噪声
F = np.stack([x1, x2, x3], axis=-1)               # (Q, M, 3)

n_tr = 150                                        # 前 150 题训练 verifier，后 150 题测试
F_tr, F_te = F[:n_tr], F[n_tr:]
y_tr, correct_te = correct[:n_tr], correct[n_tr:]
z_te = z[n_tr:]

# ---------- 手写 logistic regression ----------
def add_bias(X):
    return np.concatenate([X, np.ones(X.shape[:-1] + (1,))], axis=-1)

def logistic_nll(w, X, y):
    p = sigmoid(X @ w)
    return -np.mean(y * np.log(p + 1e-12) + (1 - y) * np.log(1 - p + 1e-12))

def logistic_grad(w, X, y):
    return X.T @ (sigmoid(X @ w) - y) / len(y)    # BCE 对 w 的精确梯度

def train_logistic(X, y, lr=0.5, steps=2000):
    w = np.zeros(X.shape[1])
    for _ in range(steps):
        w -= lr * logistic_grad(w, X, y)
    return w

w_v = train_logistic(add_bias(F_tr).reshape(-1, 4), y_tr.ravel())
v_scores = sigmoid(add_bias(F_te) @ w_v)          # (150, 64) test 题上的 verifier 概率

def auc(scores, labels):
    # Mann-Whitney U：随机抽一对 (正, 负)，正样本分数更高的概率
    order = np.argsort(scores)
    ranks = np.empty(len(scores)); ranks[order] = np.arange(1, len(scores) + 1)
    pos = labels == 1
    n_pos, n_neg = pos.sum(), (~pos).sum()
    return (ranks[pos].sum() - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg)

print(f"verifier 权重 [x1, x2, x3, bias] = {np.round(w_v, 2)}")
print(f"test AUC = {auc(v_scores.ravel(), correct_te.ravel() == 1):.3f}")
print("注意：x2（可伪造的表面特征）拿到了正权重——它在自然分布下确实有信息，")
print("但第 2 节会看到：一旦生成器开始'钻营'这类特征，BoN 的优化压力就会把它变成攻击面。")

## 2 · Verifier-BoN 与 Goodhart 倒 U；四种选择器对比

**BoN**：$N$ 条候选里选 verifier 分最高的，$\hat y=\arg\max_i v(x,y_i)$。我们对比三种 verifier 分数：

- **低噪**：$s = z + 0.3\varepsilon$ —— 接近 oracle，accuracy 随 $N$ 单调升；
- **高噪**：$s = z + 1.5\varepsilon$ —— 升得慢、早早平台；
- **可钻营**：$s = z + 0.6\varepsilon + 8\cdot\text{hack}$ —— 3% 的**错误**解答带有让 verifier 高估的伪特征（"格式工整的胡说八道"）。
  $N$ 越大，候选池里出现 hack 样本的概率 $1-(1-0.03)^N$ 越接近 1，argmax 几乎必选中它 ——
  **true accuracy 先升后降**，这就是 Gao 2022 的 Goodhart 过优化曲线在 BoN 场景的玩具版。

优化压力用 BoN 的 KL 度量：$\mathrm{KL}(\pi_{\mathrm{BoN}}\Vert\pi)=\log N-\frac{N-1}{N}$（只随 $\log N$ 增长，BoN 是温和优化器）。

随后把四种**选择器**放进同一张图：random（随机挑一条）/ majority voting（最终答案投票）/
verifier-BoN（用第 1 节训练的 verifier）/ oracle（pass@N，coverage 上限）。
majority 需要离散答案：正确解答都给出标准答案 0，错误解答从一个**集中的错误答案分布**中采样
（几何分布——常见错误高度聚集，这正是 majority 在难题上稳定选错的原因）。

In [ ]:
Ns = np.array([1, 2, 4, 8, 16, 32, 64])

def bon_accuracy(scores, correct, Ns):
    """对每个 N：前 N 条候选里取 verifier 分最高的，返回被选解答的 true accuracy"""
    rows = np.arange(scores.shape[0])
    return np.array([correct[rows, np.argmax(scores[:, :N], axis=1)].mean() for N in Ns])

def pass_at_n(correct, Ns):
    return np.array([correct[:, :N].max(axis=1).mean() for N in Ns])

# 三种 verifier 分数（直接在分数空间构造，干净隔离"噪声"与"可钻营"两种失效）
eps = rng.normal(0, 1, z_te.shape)
hack = ((rng.random(z_te.shape) < 0.03) & (correct_te == 0)).astype(float)
s_good  = z_te + 0.3 * eps
s_noisy = z_te + 1.5 * eps
s_hack  = z_te + 0.6 * eps + 8.0 * hack

acc_good, acc_noisy = bon_accuracy(s_good, correct_te, Ns), bon_accuracy(s_noisy, correct_te, Ns)
acc_hack, acc_orc   = bon_accuracy(s_hack, correct_te, Ns), pass_at_n(correct_te, Ns)
kl = np.log(Ns) - (Ns - 1) / Ns

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for s_, lab, st in [(acc_orc, "oracle (pass@N)", "k--"), (acc_good, "verifier 低噪", "C0-o"),
                    (acc_noisy, "verifier 高噪", "C1-s"), (acc_hack, "verifier 可钻营", "C3-^")]:
    axes[0].plot(Ns, s_, st, label=lab); axes[1].plot(kl, s_, st, label=lab)
axes[0].set_xscale("log", base=2); axes[0].set_xlabel("N (候选数)")
axes[1].set_xlabel(r"优化压力 KL = log N - (N-1)/N  (nats)")
for ax in axes:
    ax.set_ylabel("true accuracy"); ax.grid(alpha=0.3); ax.legend(fontsize=8)
axes[0].set_title("verifier-BoN：accuracy vs N"); axes[1].set_title("Goodhart：accuracy vs 优化压力")
plt.tight_layout(); plt.show()

i_pk = int(np.argmax(acc_hack))
print(f"可钻营 verifier 的倒 U 峰值：N* = {Ns[i_pk]}，acc = {acc_hack[i_pk]:.3f}；"
      f"N=64 时跌到 {acc_hack[-1]:.3f}")
print(f"低噪 verifier：单调升至 {acc_good[-1]:.3f}（oracle 上限 {acc_orc[-1]:.3f}）")
print("教训：BoN 的 N 是优化压力——proxy 有可被系统性选择的误差时，存在最优 N*，过之有害。")

# ---------- 四种选择器：random / majority / verifier-BoN / oracle ----------
K_WRONG = 8
p_geo = 0.5 ** np.arange(1, K_WRONG + 1); p_geo /= p_geo.sum()   # 错误答案的集中分布
wrong_ans = rng.choice(np.arange(1, K_WRONG + 1), p=p_geo, size=correct_te.shape)
ans = np.where(correct_te == 1, 0, wrong_ans).astype(int)        # 答案 0 = 标准答案

def majority_accuracy(ans, Ns):
    # 注意：bincount().argmax() 平票时偏向小编号（=答案 0），小 N 处对 majority 略有乐观偏置
    accs = []
    for N in Ns:
        preds = np.array([np.bincount(row[:N]).argmax() for row in ans])
        accs.append((preds == 0).mean())
    return np.array(accs)

acc_rand = np.full(len(Ns), correct_te.mean())                # random = 期望意义上的 pass@1
acc_maj  = majority_accuracy(ans, Ns)
acc_bon  = bon_accuracy(v_scores, correct_te, Ns)             # 第 1 节训练出的 verifier

plt.figure(figsize=(7, 4.2))
plt.plot(Ns, acc_orc, "k--", label="oracle (pass@N)")
plt.plot(Ns, acc_bon, "C0-o", label="verifier-BoN (learned)")
plt.plot(Ns, acc_maj, "C2-s", label="majority voting")
plt.plot(Ns, acc_rand, "C7:", label="random")
plt.xscale("log", base=2); plt.xlabel("N"); plt.ylabel("accuracy")
plt.title("四种选择器：coverage 兑换成 accuracy 的不同汇率"); plt.grid(alpha=0.3); plt.legend()
plt.tight_layout(); plt.show()

print(f"{'N':>4} {'random':>8} {'majority':>9} {'verifier':>9} {'oracle':>8}")
for j, N in enumerate(Ns):
    print(f"{N:>4} {acc_rand[j]:>8.3f} {acc_maj[j]:>9.3f} {acc_bon[j]:>9.3f} {acc_orc[j]:>8.3f}")
print("majority 的天花板：错误答案集中的难题上，票数最多的就是'最常见的错'——")
print("verifier 不依赖答案众数，能兑现更多 coverage（Cobbe 2021 的核心论点）。")

## 3 · 玩具 PRM：Math-Shepherd 式自动过程标签，ORM vs PRM(min)

把每条解答建模为 $T=6$ 个推理步。生成过程：每条解答有自己的"基础出错率" $e$，
且**出错风险随推理深度上升**（hazard 递增——越往后越容易错，符合误差复利直觉）；
**一步出错就永久脱轨**（`on_track` 是 `step_ok` 的累积乘积），最终答案正确 $\iff$ 全程在轨。

- **可观测步级特征**：$q_t = \text{on\_track}_t + \varepsilon$（模型能"看出"一步大概对不对，但有噪声）。
- **Math-Shepherd 标签**（Wang 2023）：$y_t = \Pr[\text{最终正确}\mid s_{1:t}]$，用 MC rollout 估计——
  从前缀出发继续生成 $K=8$ 次，数成功比例。脱轨前缀的 rollout 必败 → 标签恒 0；在轨前缀剩余步数越少标签越高
  （rollout 用群体平均出错率 $\bar e$ 模拟"让 generator 接着写"）。
- **PRM**：1 维 logistic 拟合 $(q_t \to y_t)$（soft 标签的 BCE 梯度公式不变），解答分 = $\min_t p_t$；
- **ORM**：只看整条解答的**平均**特征 $\bar q$ 拟合最终对错。

关键机制：错误多发生在**晚期**，失败解答的 $\bar q$（如 $5/6$）与成功解答（$1$）只差细微一截，
被平均稀释、淹没在噪声里；而 `min` 直接对准最弱一环——单步特征上"坏步"与"好步"差整整 1 个单位。
这就是 Lightman 2023 "PRM 在大 $N$ 端甩开 ORM" 的机制玩具版。

In [ ]:
T = 6                                                  # 每条解答 6 个推理步
hazard = np.array([0.3, 0.5, 0.8, 1.2, 1.7, 2.2])      # 出错风险随深度上升（晚期错误为主）
e_rate = rng.uniform(0.05, 0.35, (Q, M, 1))            # 每条解答的基础出错率
step_ok = (rng.random((Q, M, T)) > e_rate * hazard).astype(float)
on_track = np.cumprod(step_ok, axis=-1)                # 一步错 -> 永久脱轨
final_ok = on_track[..., -1]                           # 最终对错
q_feat = on_track + rng.normal(0, 0.8, (Q, M, T))      # 可观测步级特征
print(f"步骤世界 pass@1 = {final_ok.mean():.3f}")

def math_shepherd_labels(on_track, e_roll, n_rollouts, rng):
    """Math-Shepherd：label[..., t] = MC 估计的 P(最终正确 | 前缀 s_{1:t})。
    脱轨前缀 -> rollout 必败（on_track=0 直接清零）；
    在轨前缀 -> 单次 rollout 成功概率 = (1 - e_roll)^(剩余步数)。"""
    T = on_track.shape[-1]
    p_finish = (1 - e_roll) ** (T - 1 - np.arange(T))            # (T,)
    u = rng.random(on_track.shape + (n_rollouts,))               # (..., T, K)
    success = (u < p_finish[..., :, None]).astype(float)
    return (on_track[..., None] * success).mean(-1)   # 脱轨前缀清零，在轨前缀取成功频率

labels = math_shepherd_labels(on_track, (e_rate * hazard).mean(), n_rollouts=8, rng=rng)

i, j = 0, int(np.argmin(on_track[0].sum(-1)))          # 找一条早早脱轨的解答看看
print(f"\n示例解答 step_ok   = {step_ok[i, j].astype(int)}")
print(f"        on_track   = {on_track[i, j].astype(int)}")
print(f"        shepherd 标签 = {np.round(labels[i, j], 2)}   <- 错误步之后恒为 0")
k = int(np.argmax(final_ok[0]))
print(f"全对解答 shepherd 标签 = {np.round(labels[i, k], 2)}   <- 越接近终点剩余风险越小，标签越高")

In [ ]:
# ---------- 训练 PRM（步级）与 ORM（解答级），BoN 对比 ----------
tr, te = slice(0, n_tr), slice(n_tr, None)

# PRM：1 维特征 q_t -> shepherd soft 标签（soft 目标下 BCE 梯度公式不变）
w_prm = train_logistic(add_bias(q_feat[tr][..., None]).reshape(-1, 2), labels[tr].ravel())
p_step = sigmoid(add_bias(q_feat[te][..., None]) @ w_prm)        # (150, 64, 6) 步级分数

# ORM：整条解答的平均特征 -> 最终对错
orm_feat = q_feat.mean(-1, keepdims=True)
w_orm = train_logistic(add_bias(orm_feat[tr]).reshape(-1, 2), final_ok[tr].ravel())
orm_sc = sigmoid(add_bias(orm_feat[te]) @ w_orm)                 # (150, 64)

final_te = final_ok[te]
prm_min, prm_prod, prm_last = p_step.min(-1), p_step.prod(-1), p_step[..., -1]

acc_prm  = bon_accuracy(prm_min, final_te, Ns)
acc_orm  = bon_accuracy(orm_sc, final_te, Ns)
acc_orc2 = pass_at_n(final_te, Ns)

plt.figure(figsize=(7, 4.2))
plt.plot(Ns, acc_orc2, "k--", label="oracle (pass@N)")
plt.plot(Ns, acc_prm, "C0-o", label="PRM (min 聚合)")
plt.plot(Ns, acc_orm, "C3-s", label="ORM")
plt.xscale("log", base=2); plt.xlabel("N"); plt.ylabel("accuracy")
plt.title("过程监督 vs 结果监督：BoN 下的差距随 N 拉大"); plt.grid(alpha=0.3); plt.legend()
plt.tight_layout(); plt.show()

print(f"AUC（解答级）: PRM-min={auc(prm_min.ravel(), final_te.ravel()==1):.3f}  "
      f"ORM={auc(orm_sc.ravel(), final_te.ravel()==1):.3f}")
print(f"N=64 BoN: PRM-min={acc_prm[-1]:.3f}  PRM-prod={bon_accuracy(prm_prod, final_te, Ns)[-1]:.3f}  "
      f"PRM-last={bon_accuracy(prm_last, final_te, Ns)[-1]:.3f}  ORM={acc_orm[-1]:.3f}")
print("min 聚合抓'最弱一环'：晚期单步错误在 ORM 的平均特征里只留下 1/6 的痕迹，被噪声淹没——")
print("大 N 端 PRM 甩开 ORM（Lightman 2023 的玩具复现）。另注意 ORM 的 AUC 并不差：")
print("AUC 度量平均判别力，而 BoN 的 argmax 考验高分尾部的纯度，两者可以脱节。")

## 4 · Verifier 的校准：reliability diagram 与 ECE

verifier 分数要当**概率**用（加权投票、风险控制、拒答阈值），就必须校准：分数 0.8 的解答里应当真有 ~80% 正确。

$$\mathrm{ECE} = \sum_b \frac{n_b}{n}\,\big|\,\mathrm{acc}_b - \mathrm{conf}_b\,\big|$$

下面画第 1 节 verifier 与 PRM(min) 的 reliability diagram。注意一个结构性事实：
$\Pr[\text{全程在轨}] \le \min_t p_t$ —— **min 聚合分数天然高估成功概率**（高分端过自信），
它是好的**排序器**但不是好的**概率**；这正是"判别力"与"校准"是两张不同体检表的原因。

In [ ]:
def reliability(probs, labels, n_bins=10):
    """返回每个非空 bin 的 (平均置信度, 实际正确率, 样本数) 与 ECE"""
    edges = np.linspace(0, 1, n_bins + 1)
    idx = np.clip(np.digitize(probs, edges) - 1, 0, n_bins - 1)
    conf, acc, cnt = [], [], []
    for b in range(n_bins):
        m = idx == b
        if m.sum() == 0:
            continue
        conf.append(probs[m].mean()); acc.append(labels[m].mean()); cnt.append(m.sum())
    conf, acc, cnt = map(np.array, (conf, acc, cnt))
    ece = np.sum(cnt / cnt.sum() * np.abs(acc - conf))
    return conf, acc, cnt, ece

c1, a1, n1, ece_v = reliability(v_scores.ravel(), correct_te.ravel())
c2, a2, n2, ece_p = reliability(prm_min.ravel(), final_te.ravel())

plt.figure(figsize=(5.2, 5))
plt.plot([0, 1], [0, 1], "k--", lw=1, label="完美校准")
plt.plot(c1, a1, "C0-o", label=f"verifier (ECE={ece_v:.3f})")
plt.plot(c2, a2, "C3-s", label=f"PRM min (ECE={ece_p:.3f})")
plt.xlabel("置信度（verifier 分数）"); plt.ylabel("实际正确率")
plt.title("reliability diagram"); plt.grid(alpha=0.3); plt.legend()
plt.tight_layout(); plt.show()

print(f"verifier ECE = {ece_v:.3f}（直接以 BCE 训练，校准较好）")
print(f"PRM-min  ECE = {ece_p:.3f}（min 不是概率：判别力强 != 校准好，两张体检表要分开看）")

---
## ✏️ 练习 1：手写 logistic 梯度 + 数值梯度校验

实现 `logistic_grad_ex(w, X, y)`：返回平均 BCE 损失 $-\frac1n\sum_i[y_i\log p_i+(1-y_i)\log(1-p_i)]$
（$p_i=\sigma(x_i^\top w)$）对 $w$ 的梯度。

**提示**：sigmoid + BCE 的梯度有著名的简洁形式 $\frac1n X^\top(p-y)$ —— 推一遍链式法则确认。
1~2 行即可。自测会用中心差分数值梯度 $\frac{L(w+\epsilon e_j)-L(w-\epsilon e_j)}{2\epsilon}$ 逐维比对
（这是检查任何手写梯度的标准武器），边界：梯度形状必须与 `w` 一致。

In [ ]:
def logistic_grad_ex(w, X, y):
    # TODO: p = sigmoid(X @ w)，返回平均 BCE 对 w 的梯度（形状与 w 相同）
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
rng_t = np.random.default_rng(0)
Xt = rng_t.normal(size=(50, 3))
yt = (rng_t.random(50) < sigmoid(Xt @ np.array([1.0, -2.0, 0.5]))).astype(float)
wt = rng_t.normal(size=3)

g = logistic_grad_ex(wt, Xt, yt)
assert g.shape == (3,)
num = np.zeros(3)
for j in range(3):
    e = np.zeros(3); e[j] = 1e-6
    num[j] = (logistic_nll(wt + e, Xt, yt) - logistic_nll(wt - e, Xt, yt)) / 2e-6
assert np.max(np.abs(g - num)) < 1e-5, f"与数值梯度不符: {g} vs {num}"
assert np.allclose(logistic_grad_ex(np.zeros(3), Xt, yt), Xt.T @ (0.5 - yt) / 50)  # w=0 时 p=0.5
print("✅ 练习 1 通过")

## ✏️ 练习 2：`bon_select_ex` 与 Goodhart 倒 U

实现 `bon_select_ex(scores, correct, N)`：`scores`/`correct` 形状 `(Q, M)`，
每题在**前 N 条**候选中选分数最高的一条，返回被选解答的平均 true accuracy（标量）。

**提示**：`np.argmax(scores[:, :N], axis=1)` 拿到每题选中的列号，再用
`correct[np.arange(Q), sel]` 花式索引取对错。3~5 行。
自测验证两个理论性质：① 分数=真实质量（零噪声）时曲线**单调不减**且大 N 端到 1.0
（前缀最大值随 N 单调 → 逐题指标单调）；② 注入强伪特征（少量错误解答 +10 分）后出现**内点峰值**——
峰不在 N=1 也不在 N=64，且大 N 端明显跌落。

In [ ]:
def bon_select_ex(scores, correct, N):
    # TODO: 每题在前 N 条候选里取 argmax(scores)，返回被选中解答的 correct 平均值
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
rng_t = np.random.default_rng(0)
z_t = rng_t.normal(size=(400, 64))
corr_t = (z_t > 0).astype(float)
Ns_t = [1, 2, 4, 8, 16, 32, 64]

acc0 = np.array([bon_select_ex(z_t, corr_t, N) for N in Ns_t])       # 零噪声 verifier
assert np.all(np.diff(acc0) >= -1e-12), "零噪声时应单调不减"
assert acc0[-1] == 1.0, "64 条里挑真实质量最高的，应全对"

hack_t = (rng_t.random((400, 64)) < 0.04) & (corr_t == 0)            # 4% 错误解答可钻营
acc1 = np.array([bon_select_ex(z_t + 10.0 * hack_t, corr_t, N) for N in Ns_t])
k = int(np.argmax(acc1))
assert 0 < k < len(Ns_t) - 1, "伪特征强时峰值应是内点（先升后降）"
assert acc1[-1] < acc1[k] - 0.1, "大 N 端应明显跌落（Goodhart）"
assert acc1[k] > acc1[0], "小 N 端仍应先上升"
print("✅ 练习 2 通过")

## ✏️ 练习 3：`math_shepherd_labels_ex` —— 单条解答的 MC 过程标签

实现 `math_shepherd_labels_ex(step_ok, e_roll, n_rollouts, rng)`：输入一条解答的步骤对错
`step_ok`（长度 $T$ 的 0/1 数组），返回长度 $T$ 的标签数组，
`label[t]` = MC 估计的 $\Pr[\text{最终正确}\mid s_{1:t}]$。

**提示**：先算 `on_track = np.cumprod(step_ok)`；脱轨前缀直接 0；在轨前缀做 `n_rollouts` 次伯努利试验，
单次成功概率 $(1-e_\text{roll})^{T-1-t}$（剩余步全对），取成功频率。约 6~8 行。
边界：最后一步若在轨，剩余 0 步 → 标签应恰为 1.0；错误步**及其之后**标签全 0。

In [ ]:
def math_shepherd_labels_ex(step_ok, e_roll, n_rollouts, rng):
    # TODO:
    #   1) on_track = np.cumprod(step_ok)
    #   2) 对每个 t：p_finish = (1 - e_roll) ** (T - 1 - t)
    #   3) label[t] = on_track[t] * (rng.random(n_rollouts) < p_finish).mean()
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
rng_t = np.random.default_rng(0)

lab = math_shepherd_labels_ex(np.array([1, 1, 0, 1, 1, 1]), 0.2, 200, rng_t)
assert lab.shape == (6,)
assert np.all(lab[2:] == 0), "错误步及其之后，rollout 必败 -> 标签全 0"
assert np.all(lab[:2] > 0), "正确前缀总有活路 -> 标签 > 0"

lab2 = math_shepherd_labels_ex(np.ones(6, dtype=int), 0.0, 50, rng_t)
assert np.allclose(lab2, 1.0), "全对解答 + rollout 零出错率 -> 标签全 1"

lab3 = math_shepherd_labels_ex(np.ones(6, dtype=int), 0.3, 400, rng_t)
assert lab3[-1] == 1.0, "最后一步在轨且无剩余步骤 -> 标签恰为 1"
assert lab3[0] < lab3[-1], "越接近终点剩余风险越小 -> 标签应上升"
assert np.all((lab3 >= 0) & (lab3 <= 1))
print("✅ 练习 3 通过")

## ✏️ 练习 4：`prm_vs_orm_ex` —— min 聚合 PRM 对 ORM 的 BoN 对比

实现 `prm_vs_orm_ex(step_probs, orm_scores, correct, Ns)`：
`step_probs` 形状 `(Q, M, T)` 是 PRM 步级分数，先做 **min 聚合**得到解答分，
再与解答级 `orm_scores`（形状 `(Q, M)`）各自跑 BoN，返回 `(acc_prm, acc_orm)` 两个数组。

**提示**：min 聚合一行 `step_probs.min(axis=-1)`；BoN 直接复用上文 `bon_accuracy`（或练习 2 的实现）。
3~4 行。自测用第 3 节的真实数据（`p_step` / `orm_sc` / `final_te`），
验证：与主线结果一致、PRM 在大 N 端 ≥ ORM、BoN 相对 N=1 有正收益。

In [ ]:
def prm_vs_orm_ex(step_probs, orm_scores, correct, Ns):
    # TODO: prm_solution = step_probs 沿最后一维取 min；
    #       分别对 prm_solution 与 orm_scores 跑 BoN，返回 (acc_prm, acc_orm)
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
a_prm, a_orm = prm_vs_orm_ex(p_step, orm_sc, final_te, Ns)
assert len(a_prm) == len(Ns) and len(a_orm) == len(Ns)
assert np.allclose(a_prm, bon_accuracy(p_step.min(-1), final_te, Ns)), "min 聚合或 BoN 实现不对"
assert np.allclose(a_orm, bon_accuracy(orm_sc, final_te, Ns))
assert a_prm[-1] >= a_orm[-1] + 0.02, "大 N 端 PRM(min) 应明显优于 ORM"
assert np.mean(a_prm) >= np.mean(a_orm), "整体上过程监督不应输给结果监督"
assert a_prm[-1] > a_prm[0], "BoN 相对 N=1 应有正收益"
print("✅ 练习 4 通过")

---
## 📖 参考答案

In [ ]:
# 练习 1 参考答案（先自己做，再对照）
def logistic_grad_ex(w, X, y):
    return X.T @ (sigmoid(X @ w) - y) / len(y)

In [ ]:
# 练习 2 参考答案（先自己做，再对照）
def bon_select_ex(scores, correct, N):
    sel = np.argmax(scores[:, :N], axis=1)
    return correct[np.arange(scores.shape[0]), sel].mean()

In [ ]:
# 练习 3 参考答案（先自己做，再对照）
def math_shepherd_labels_ex(step_ok, e_roll, n_rollouts, rng):
    T = len(step_ok)
    on_track = np.cumprod(step_ok)
    labels = np.zeros(T, dtype=float)
    for t in range(T):
        if on_track[t]:
            p_finish = (1 - e_roll) ** (T - 1 - t)
            labels[t] = (rng.random(n_rollouts) < p_finish).mean()
    return labels

In [ ]:
# 练习 4 参考答案（先自己做，再对照）
def prm_vs_orm_ex(step_probs, orm_scores, correct, Ns):
    prm_solution = step_probs.min(axis=-1)
    return bon_accuracy(prm_solution, correct, Ns), bon_accuracy(orm_scores, correct, Ns)

---
## 🎯 真实数据胶囊题：真实 GSM8K 上的 best-of-N 与验证器增益

有一个(带噪的)验证器给候选打分、选最高分，best-of-N 能超过随机选。用真实 GSM8K，模拟 N 个候选(部分正确)+ 噪声验证器，验证 best-of-N 准确率高于随机选一个。

> 本模块新增的**真实数据**练习：自包含、用真实 GSM8K 把本章方法跑一遍。先做 TODO，`assert` 全过即通关，文末有参考答案。

In [ ]:
import os, json, urllib.request, re
import numpy as np
CACHE=os.path.expanduser("~/.reasoning_ttc_data"); os.makedirs(CACHE,exist_ok=True)
def _f(url,fn):
    p=os.path.join(CACHE,fn)
    if not os.path.exists(p): urllib.request.urlretrieve(url,p)
    return p
def gsm8k(n=300):
    p=_f("https://raw.githubusercontent.com/openai/grade-school-math/master/grade_school_math/data/test.jsonl","gsm8k_test.jsonl")
    return [json.loads(l) for l in open(p).read().splitlines()[:n]]
def gold(a): return a.split("####")[-1].strip().replace(",","")
def steps(a): return max(1, a.count("<<"))   # 真实推理步数代理

rows=gsm8k(150); golds=[gold(r["answer"]) for r in rows]
rng=np.random.default_rng(0)
def candidates(g, N=8, p=0.4):
    cands=[g if rng.random()<p else str(int(rng.integers(0,99999))) for _ in range(N)]
    correct=[c==g for c in cands]
    # 验证器分数：对的给高分(带噪)，错的给低分(带噪)
    scores=[ (1.0 if ok else 0.0) + rng.normal(0,0.4) for ok in correct]
    return cands, np.array(scores), correct

**练习**：实现 `best_of_n_acc(golds, N, p, seed)`：每题生成 N 候选+验证器分，选**分最高**的候选，返回准确率。应高于随机选一个候选的准确率(≈p)。

In [ ]:
def best_of_n_acc(golds, N=8, p=0.4, seed=0):
    # TODO: 每题 candidates() -> 选 score 最大的候选 -> 与 gold 比
    raise NotImplementedError


In [ ]:
# 自测
acc=best_of_n_acc(golds, 8, 0.4, seed=1)
assert acc > 0.4, f"best-of-8 应高于单候选正确率0.4, 得到{acc:.2f}"
print(f"best-of-8 准确率 = {acc:.2f} > 单候选 0.40 ✓ (验证器把好答案捞出来)")


### 📖 参考答案

In [ ]:
def best_of_n_acc(golds, N=8, p=0.4, seed=0):
    rng2=np.random.default_rng(seed); c=0
    for g in golds:
        cands=[g if rng2.random()<p else str(int(rng2.integers(0,99999))) for _ in range(N)]
        ok=[x==g for x in cands]
        sc=np.array([(1.0 if o else 0.0)+rng2.normal(0,0.4) for o in ok])
        if cands[sc.argmax()]==g: c+=1
    return c/len(golds)
print("✓ best-of-N + verifier 是测试时计算最简单有效的一招")